# 03. Select Labeling Candidate Tiles

Rank 1024 x 1024 tiles by a simple HSV vegetation and texture heuristic, then copy the strongest full-tile candidates into a labeling folder.

**Input**
- `data/processed/tiles_1024/images/*.png`
- `data/processed/tiles_1024/tile_index.csv`

**Output**
- `data/processed/labeling_candidates_1024/images/*.png`
- `data/processed/labeling_candidates_1024/all_tile_scores.csv`
- `data/processed/labeling_candidates_1024/selected_tiles.csv`
- `data/processed/labeling_candidates_1024/preview/selected_contact_sheet.png`

This is a first-pass helper, not a tree detector. Review the selected tiles visually before polygon labeling. Edge tiles are excluded by default because padded regions can make labeling awkward.

In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
import numpy as np
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path("../..").resolve()

TILE_ROOT = PROJECT_ROOT / "data/processed/tiles_1024"
TILE_IMAGE_DIR = TILE_ROOT / "images"
TILE_INDEX_PATH = TILE_ROOT / "tile_index.csv"

OUTPUT_ROOT = PROJECT_ROOT / "data/processed/labeling_candidates_1024"
OUTPUT_IMAGE_DIR = OUTPUT_ROOT / "images"
PREVIEW_DIR = OUTPUT_ROOT / "preview"
ALL_SCORES_PATH = OUTPUT_ROOT / "all_tile_scores.csv"
SELECTED_PATH = OUTPUT_ROOT / "selected_tiles.csv"
CONTACT_SHEET_PATH = PREVIEW_DIR / "selected_contact_sheet.png"

TOP_N = 12
EXCLUDE_EDGE_TILES = True

OUTPUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

if not TILE_INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing tile index: {TILE_INDEX_PATH}")

index = pd.read_csv(TILE_INDEX_PATH)
print(f"Tiles loaded: {len(index)}")
print(f"Candidate output: {OUTPUT_ROOT}")

In [ ]:
def score_tile(image_path: Path, valid_width: int, valid_height: int) -> dict:
    arr = np.array(Image.open(image_path).convert("RGB"), dtype=np.float32)
    valid = arr[:valid_height, :valid_width, :]

    r = valid[..., 0]
    g = valid[..., 1]
    b = valid[..., 2]
    brightness = valid.mean(axis=2)

    # HSV green with moderate saturation is stricter than raw excess green.
    hsv = rgb_to_hsv(valid / 255.0)
    hue = hsv[..., 0]
    saturation = hsv[..., 1]
    value = hsv[..., 2]
    green_mask = (hue > 0.20) & (hue < 0.42) & (saturation > 0.18) & (value > 0.20)

    # Excess green is kept as a secondary signal.
    exg = 2 * g - r - b

    grad_y, grad_x = np.gradient(brightness)
    texture = np.sqrt(grad_x ** 2 + grad_y ** 2)

    vegetation_ratio = float(green_mask.mean())
    green_texture_mean = float(texture[green_mask].mean()) if green_mask.any() else 0.0
    texture_mean = float(texture.mean())
    dark_ratio = float((brightness < 35).mean())
    bright_ratio = float((brightness > 245).mean())
    mean_exg = float(exg.mean())

    score = (
        vegetation_ratio * 100
        + min(green_texture_mean, 18) * 1.25
        + min(texture_mean, 18) * 0.35
        + max(mean_exg, 0) * 0.02
        - dark_ratio * 6
        - bright_ratio * 4
    )

    return {
        "vegetation_ratio": vegetation_ratio,
        "green_texture_mean": green_texture_mean,
        "texture_mean": texture_mean,
        "dark_ratio": dark_ratio,
        "bright_ratio": bright_ratio,
        "mean_exg": mean_exg,
        "candidate_score": float(score),
    }


scores = []
for _, rec in index.iterrows():
    image_path = TILE_IMAGE_DIR / rec["file_name"]
    if not image_path.exists():
        raise FileNotFoundError(f"Missing tile image: {image_path}")

    metrics = score_tile(
        image_path,
        valid_width=int(rec["valid_width"]),
        valid_height=int(rec["valid_height"]),
    )
    scores.append({**rec.to_dict(), **metrics})

scores_df = pd.DataFrame(scores).sort_values("candidate_score", ascending=False).reset_index(drop=True)
scores_df["score_rank"] = range(1, len(scores_df) + 1)
scores_df.to_csv(ALL_SCORES_PATH, index=False)

scores_df[["score_rank", "tile_id", "vegetation_ratio", "green_texture_mean", "texture_mean", "candidate_score", "is_edge"]].head(15)

In [ ]:
eligible = scores_df.copy()
if EXCLUDE_EDGE_TILES:
    eligible = eligible[~eligible["is_edge"].astype(bool)].copy()

selected = eligible.head(TOP_N).copy()
selected["selection_rank"] = range(1, len(selected) + 1)
selected.to_csv(SELECTED_PATH, index=False)

for old_file in OUTPUT_IMAGE_DIR.glob("*.png"):
    old_file.unlink()

for _, rec in selected.iterrows():
    src = TILE_IMAGE_DIR / rec["file_name"]
    dst = OUTPUT_IMAGE_DIR / f"rank{int(rec['selection_rank']):02d}_{rec['file_name']}"
    shutil.copy2(src, dst)

print(f"Selected tiles: {len(selected)}")
print(f"Excluded edge tiles: {EXCLUDE_EDGE_TILES}")
print(f"Saved selected tile CSV: {SELECTED_PATH}")
print(f"Copied selected images to: {OUTPUT_IMAGE_DIR}")
selected[["selection_rank", "score_rank", "tile_id", "file_name", "vegetation_ratio", "green_texture_mean", "texture_mean", "candidate_score", "is_edge"]]

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(selected) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3.4))
axes = np.array(axes).reshape(n_rows, n_cols)

for ax in axes.ravel():
    ax.axis("off")

for ax, (_, rec) in zip(axes.ravel(), selected.iterrows()):
    img = Image.open(TILE_IMAGE_DIR / rec["file_name"])
    ax.imshow(img)
    ax.set_title(
        f"#{int(rec['selection_rank'])} {rec['tile_id']}\nveg={rec['vegetation_ratio']:.2f}, score={rec['candidate_score']:.1f}",
        fontsize=9,
    )

plt.tight_layout()
fig.savefig(CONTACT_SHEET_PATH, dpi=150)
plt.show()

print(f"Saved selected contact sheet: {CONTACT_SHEET_PATH}")